# Tester le modele sur un NOUVEAU texte d'espece

Charge le modele entraine (`outputs/bio/bert/baseline/final_model/`) et extrait les **traits** (longueurs, masses, ages, temperature...) d'une description d'espece **jamais vue** par le modele.

Deux affichages : un **tableau** des entites trouvees, et le **texte surligne** en couleur.

Requis : `torch`, `transformers`, `pandas`. Lance : **Kernel -> Restart & Run All**.

In [1]:
import html as _html
import re
from pathlib import Path

import pandas as pd
import torch
from IPython.display import HTML, display
from transformers import AutoModelForTokenClassification, AutoTokenizer

ROOT = Path.cwd()
while not (ROOT / "outputs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MODEL_DIR = ROOT / "outputs" / "bio" / "bert" / "baseline" / "final_model"

device = "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR)).eval().to(device)
id2label = {int(k): v for k, v in model.config.id2label.items()}
MAX_LEN = min(tokenizer.model_max_length, model.config.max_position_embeddings)

print("Modele :", MODEL_DIR.name, "| device :", device, "|", len(id2label), "labels")

Modele : final_model | device : mps | 41 labels


### Fonctions (tokenisation identique a l'entrainement + decodage BIO)

In [2]:
_SENT = re.compile(r"(?<=[.!?])\s+")
# garde les nombres decimaux entiers ("1.5", "2,5") et separe la ponctuation
_WORD = re.compile(r"\d+(?:[.,]\d+)*|\w+|[^\w\s]")


def _tokenize(sentence):
    words, offsets = [], []
    for m in _WORD.finditer(sentence):
        words.append(m.group())
        offsets.append((m.start(), m.end()))
    return words, offsets


def _spans(tags, offsets):
    """BIO -> spans. Un I- orphelin (sans B- du meme type) est ignore."""
    spans, cur = [], None
    for tag, (start, end) in zip(tags, offsets):
        if tag == "O":
            if cur: spans.append(cur); cur = None
            continue
        prefix, etype = tag.split("-", 1)
        if prefix == "B":
            if cur: spans.append(cur)
            cur = {"type": etype, "start": start, "end": end}
        elif cur and cur["type"] == etype:
            cur["end"] = end
    if cur: spans.append(cur)
    return spans


def analyse(text):
    """-> liste de (phrase, [spans])"""
    out = []
    for sentence in [s for s in _SENT.split(text.strip()) if s]:
        words, offsets = _tokenize(sentence)
        if not words:
            out.append((sentence, []))
            continue
        enc = tokenizer(words, is_split_into_words=True, return_tensors="pt",
                        truncation=True, max_length=MAX_LEN)
        word_ids = enc.word_ids(batch_index=0)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            pred_ids = model(**enc).logits[0].argmax(-1).tolist()
        seen, tags, offs = set(), [], []
        for i, wid in enumerate(word_ids):
            if wid is None or wid in seen:
                continue
            seen.add(wid)
            tags.append(id2label[pred_ids[i]])
            offs.append(offsets[wid])
        out.append((sentence, _spans(tags, offs)))
    return out


def table(text):
    rows = [{"Type": sp["type"], "Texte extrait": sent[sp["start"]:sp["end"]], "Phrase": sent}
            for sent, spans in analyse(text) for sp in spans]
    return pd.DataFrame(rows)


_PALETTE = ["#c6dbef", "#c7e9c0", "#fdd0a2", "#dadaeb", "#fdd49e", "#d9f0d3",
            "#fde0dd", "#e5f5e0", "#bcbddc", "#fdae6b"]


def highlight(text):
    colors, blocks = {}, []
    for sent, spans in analyse(text):
        pieces, last = [], 0
        for sp in sorted(spans, key=lambda s: s["start"]):
            if sp["start"] < last:
                continue
            col = colors.setdefault(sp["type"], _PALETTE[len(colors) % len(_PALETTE)])
            pieces.append(_html.escape(sent[last:sp["start"]]))
            frag = _html.escape(sent[sp["start"]:sp["end"]])
            pieces.append(
                f'<mark style="background:{col};color:#111;padding:1px 4px;border-radius:4px">'
                f'{frag}<sup style="font-size:.7em;opacity:.7"> {sp["type"]}</sup></mark>'
            )
            last = sp["end"]
        pieces.append(_html.escape(sent[last:]))
        blocks.append("".join(pieces))
    return HTML("<div style='line-height:2.1;font-size:15px'>" + " ".join(blocks) + "</div>")

### Nouveau texte d'espece (jamais vu par le modele)

Le lapin europeen (*Oryctolagus cuniculus*) — description contenant plusieurs traits : gestation, masse a la naissance, sevrage, maturite, longueur adulte, masse adulte, longevite, temperature.

In [3]:
NEW_TEXT = (
    "The European rabbit (Oryctolagus cuniculus) is a small burrowing mammal native to the Iberian "
    "Peninsula and now widely introduced across temperate regions. Females give birth after a gestation "
    "period of about 30 days, and newborn kits weigh approximately 45 g. The young are weaned at around "
    "25 days and reach sexual maturity at about 4 months of age. Adults typically attain a body length of "
    "38 to 50 cm and a body mass of 1.5 to 2.5 kg. In the wild, individuals rarely survive beyond 9 years, "
    "although captive animals have been recorded living up to 12 years. Colonies thrive in habitats where "
    "the ambient temperature averages around 15 degrees C throughout the year."
)

df = table(NEW_TEXT)
print(f"{len(df)} entites extraites  |  types : {sorted(df['Type'].unique())}")
display(df[["Type", "Texte extrait"]])

5 entites extraites  |  types : ['Li', 'T', 'Wwb', 'ab', 'tx']


,Type,Texte extrait
0,ab,about 30 days
1,Wwb,approximately 45
2,tx,around 25 days
3,Li,38 to 50 cm
4,T,15 degrees C


### Le texte surligne

In [4]:
highlight(NEW_TEXT)

### Compte par type de trait

In [5]:
counts = df["Type"].value_counts().rename_axis("Type").reset_index(name="Nombre")
display(counts)

,Type,Nombre
0,ab,1
1,Wwb,1
2,tx,1
3,Li,1
4,T,1


### A toi : colle ton propre texte ici

Remplace le texte ci-dessous par n'importe quelle description d'espece (en anglais, comme les donnees d'entrainement).

In [6]:
MY_TEXT = (
    "Colle ici ta propre description d'espece. "
    "Par exemple : Adults reach a total length of 24 cm and weigh about 180 g. "
    "Females lay eggs after an incubation period of 18 days, and hatchlings weigh 3.2 g."
)

my_df = table(MY_TEXT)
print(f"{len(my_df)} entites extraites")
display(my_df[["Type", "Texte extrait"]])
display(highlight(MY_TEXT))

2 entites extraites


,Type,Texte extrait
0,ab,18 days
1,Wwb,3.2
